# ShallKnow – small-model inference benchmark

This notebook benchmarks the **LLM simplification step** for the selected models using a small, fixed sample of Wikipedia pages and entity pairs. It follows the relevant parts of `wikipedia_extractor_final.py`:

1. Load only reified RDF statements whose `hasTruthValue` is `1.0`.
2. Fetch the subject/object Wikipedia pages and a bounded number of their outgoing links.
3. Apply the same text-cleaning rules as the extractor.
4. Build the same entity-pair relationship prompt.
5. Send the **same prompts** to every model and record latency, token counts, output size, and errors.

It deliberately does **not** run the full extractor or write its processed-title/hash bookkeeping files. This makes it safe to run as a benchmark without modifying the real experiment output.

**Important:** this measures LLM inference separately from Wikipedia download and Maverick coreference. There is an optional coreference timing section near the end. The first request for each model may include model-loading time; the notebook reports first-call and subsequent-call timings separately.

Run from the repository root, with the project virtual environment active and Ollama (or your compatible API gateway) available.


## 1. Configuration

Adjust the paths and sample size below. For a useful initial estimate, try around **20–50 prompts** per model. A tiny sample (e.g. 3 prompts) is useful for checking that everything works, but is not a reliable estimate.

The model tags below are the team's selected lineup. Confirm they exist on the machine with `ollama list` and pull them before starting.


In [ ]:
import os
from pathlib import Path

# ADRES_MODELU z pliku .env w katalogu repo (plik jest w .gitignore), jeśli nie ma go w środowisku
_env = Path.cwd().parent / ".env"
if "ADRES_MODELU" not in os.environ and _env.exists():
    for line in _env.read_text().splitlines():
        if line.startswith("ADRES_MODELU="):
            os.environ["ADRES_MODELU"] = line.split("=", 1)[1].strip()

print("token:", "jest" if os.environ.get("TOKEN_MODELU") else "BRAK")
print("adres:", "jest" if os.environ.get("ADRES_MODELU") else "BRAK (użyje localhost:11434)")


In [3]:
from pathlib import Path
import os
import re
import html
import json
import time
import hashlib
import random
import statistics
from datetime import datetime, timezone

import requests
import pandas as pd
from rdflib import Graph, URIRef
from rdflib.namespace import RDF
from IPython.display import display

import time

# ---- Paths ----
PROJECT_ROOT = Path.cwd().parent
INPUT_FILE = PROJECT_ROOT / "data" / "input" / "BPDP_train_reified.nt"
# If your prepared .nt file is elsewhere, change INPUT_FILE.

# ---- Models: update tags if your Ollama installation uses different names ----
MODELS = [
    "qwen3.5:2b",
    "qwen3.5:4b",
    "gemma4:e2b",
    "gemma4:e4b",
    "qwen3.5:9b",
    "deepseek-r1:14b"
]

# ---- Sampling ----
N_TRIPLES_TO_SCAN = 30       # scan this many true triples to find enough usable pages
N_PROMPTS = 30               # number of unique page/entity-pair prompts benchmarked per model
MAX_LINKS_PER_ENTITY_PAGE = 10  # keep this small; extractor can encounter hundreds of links
RANDOM_SEED = 42

# ---- Inference ----
KEEP_ALIVE = "10m"
TEMPERATURE = 0
NUM_PREDICT = 4096            # output-token budget; check done_reason for "length"
NUM_CTX = None                # e.g. 8192; None leaves server/model default unchanged
THINK_MODES = [True, False]   # None = omit parameter and use model default; True/False if supported

# ---- Output ----
RESULTS_DIR = PROJECT_ROOT / "results" / "model_benchmark"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
print("Project root:", PROJECT_ROOT)
print("Input exists:", INPUT_FILE.exists(), "|", INPUT_FILE)
print("Results will be saved to:", RESULTS_DIR)


Project root: /Users/michal/university/sem7/semantic-web/no-need-to-be-so-big
Input exists: True | /Users/michal/university/sem7/semantic-web/no-need-to-be-so-big/data/input/BPDP_train_reified.nt
Results will be saved to: /Users/michal/university/sem7/semantic-web/no-need-to-be-so-big/results/model_benchmark


## 2. Check the inference endpoint

The notebook supports local Ollama and the project's remote gateway environment variables:

- `OLLAMA_URL`: full `/api/generate` URL, if explicitly set.
- `ADRES_MODELU`: server base URL; `/api/generate` is appended automatically.
- `TOKEN_MODELU`: optional API key sent as `X-Api-Key`.

For local Ollama, the default is `http://localhost:11434/api/generate`.


In [4]:
def resolve_ollama_endpoint():
    explicit = os.environ.get("OLLAMA_URL")
    if explicit:
        url = explicit.rstrip("/")
        return url if url.endswith("/api/generate") else url + "/api/generate"

    base = os.environ.get("ADRES_MODELU", "http://localhost:11434").rstrip("/")
    if base.endswith("/api/generate"):
        return base
    return base + "/api/generate"

API_URL = resolve_ollama_endpoint()
API_HEADERS = {}
if os.environ.get("TOKEN_MODELU"):
    API_HEADERS["X-Api-Key"] = os.environ["TOKEN_MODELU"]

print("API endpoint:", API_URL)
print("API key configured:", bool(API_HEADERS))

# Basic connectivity check. This does not load a model.
try:
    r = requests.get(API_URL.rsplit("/api/generate", 1)[0] + "/api/tags", timeout=15)
    print("Ollama tags endpoint HTTP status:", r.status_code)
    if r.ok:
        installed = [m.get("name") for m in r.json().get("models", [])]
        print("Installed models:", installed)
except requests.RequestException as e:
    print("Could not query /api/tags:", repr(e))
    print("If using a gateway that does not expose /api/tags, this may be expected.")


API endpoint: <ADRES_MODELU>/api/generate
API key configured: True
Ollama tags endpoint HTTP status: 401


## 3. Load true RDF statements and reproduce the extractor's text cleaning

The extractor filters to `hasTruthValue == "1.0"`. The helper functions below mirror its RDF loading, title extraction, Wikipedia retrieval, and `clean_text()` behavior. Wikipedia link retrieval includes pagination, as in the base file.


In [5]:
def load_true_triples(path):
    graph = Graph()
    graph.parse(str(path), format="nt")
    truth_predicate = URIRef("http://swc2017.aksw.org/hasTruthValue")
    triples = []

    for stmt in graph.subjects(RDF.type, RDF.Statement):
        truth_value = graph.value(stmt, truth_predicate)
        if truth_value is None or str(truth_value) != "1.0":
            continue
        subj = graph.value(stmt, RDF.subject)
        pred = graph.value(stmt, RDF.predicate)
        obj = graph.value(stmt, RDF.object)
        if subj and pred and obj:
            triples.append((str(subj), str(pred), str(obj)))

    return sorted(triples)


def title_from_uri(uri):
    return uri.rsplit("/", 1)[-1]


def normalize_filename(filename):
    filename = re.sub(r'[\\\\/:"*?<>|]', "_", filename)
    return filename.replace("_", " ").strip().lower()


def clean_text(text):
    """
    Mirrors clean_text() in wikipedia_extractor_final.py.
    """
    text = text.encode("utf-8").decode("utf-8")
    text = html.unescape(text)
    text = re.sub(r'={2,}.*?={2,}', '', text)
    text = re.sub(r'\[\[.*?\|(.*?)\]\]', r'\1', text)
    text = re.sub(r'\[\[(.*?)\]\]', r'\1', text)
    text = re.sub(r'<ref.*?>.*?</ref>', '', text, flags=re.DOTALL)
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'\{\{.*?\}\}', '', text)
    text = text.replace("\\", "").replace("'", "").replace('"', '')
    text = re.sub(r'\(\s*[^()]*[^\x00-\x7F]+[^()]*\s*\)', '', text)
    text = re.sub(r'\(\s*\)', '', text)
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = re.sub(r'(?<!\n)\n(?!\n)', '\n\n', text)
    return text

WIKI_HEADERS = {
    "User-Agent": (
        "no-need-to-be-so-big/0.1 "
        "(university project; https://github.com/ZiarnoM/no-need-to-be-so-big)"
    )
}
WIKI_API = "https://en.wikipedia.org/w/api.php"
WIKI_SESSION = requests.Session()
WIKI_CACHE = {}
LAST_WIKI_REQUEST = 0.0
MIN_REQUEST_INTERVAL = 1.2

def wiki_get(params, max_retries=6):
    global LAST_WIKI_REQUEST

    params = dict(params)
    params["maxlag"] = 5

    for attempt in range(max_retries):
        wait = MIN_REQUEST_INTERVAL - (time.monotonic() - LAST_WIKI_REQUEST)
        if wait > 0:
            time.sleep(wait)

        response = WIKI_SESSION.get(
            WIKI_API,
            params=params,
            headers=WIKI_HEADERS,
            timeout=45
        )
        LAST_WIKI_REQUEST = time.monotonic()

        if response.status_code == 429 or response.status_code in (500, 502, 503, 504):
            if attempt == max_retries - 1:
                response.raise_for_status()

            retry_after = response.headers.get("Retry-After")
            delay = float(retry_after) if retry_after and retry_after.isdigit() else min(60, 5 * (2 ** attempt))
            print(f"Wikipedia rate limit/server error. Waiting {delay}s...")
            time.sleep(delay)
            continue

        response.raise_for_status()
        return response.json()

    raise RuntimeError("Wikipedia request failed after retries")

def fetch_wikipedia_content(title, include_links=True, max_links=None):
    cache_key = (title, include_links, max_links)

    if cache_key in WIKI_CACHE:
        return WIKI_CACHE[cache_key]

    params = {
        "action": "query",
        "format": "json",
        "prop": "extracts|links" if include_links else "extracts",
        "explaintext": 1,
        "redirects": 1,
        "titles": title
    }

    if include_links:
        params["pllimit"] = "max"

    data = wiki_get(params)
    query = data.get("query", {})

    if query.get("redirects"):
        redirected = query["redirects"][0]["to"]
        return fetch_wikipedia_content(
            redirected, include_links, max_links
        )

    pages = query.get("pages", {})
    page = next(iter(pages.values()), {})

    if not page or "missing" in page:
        return None, []

    text = page.get("extract", "")
    links = [
        x["title"] for x in page.get("links", [])
        if "title" in x
    ]

    while (
        include_links
        and "continue" in data
        and "plcontinue" in data["continue"]
        and (max_links is None or len(links) < max_links)
    ):
        params["plcontinue"] = data["continue"]["plcontinue"]
        data = wiki_get(params)

        for p in data.get("query", {}).get("pages", {}).values():
            links.extend(
                x["title"] for x in p.get("links", [])
                if "title" in x
            )

        if max_links is not None:
            links = links[:max_links]

    result = (text, links)
    WIKI_CACHE[cache_key] = result
    return result


if not INPUT_FILE.exists():
    raise FileNotFoundError(
        f"Could not find {INPUT_FILE}. Update INPUT_FILE in the configuration cell."
    )

triples = load_true_triples(INPUT_FILE)
print(f"Loaded {len(triples):,} true statements.")
display(pd.DataFrame(triples[:5], columns=["subject_uri", "predicate_uri", "object_uri"]))


Loaded 100 true statements.


,subject_uri,predicate_uri,object_uri
0,http://dbpedia.org/resource/A._E._van_Vogt,http://www.wikidata.org/prop/direct/P19,http://dbpedia.org/resource/Winnipeg
1,http://dbpedia.org/resource/A._E._van_Vogt,http://www.wikidata.org/prop/direct/P20,http://dbpedia.org/resource/Hollywood
2,http://dbpedia.org/resource/Adam_Mickiewicz,http://www.wikidata.org/prop/direct/P19,http://dbpedia.org/resource/Lithuania_Governorate
3,http://dbpedia.org/resource/Adam_Mickiewicz,http://www.wikidata.org/prop/direct/P20,http://dbpedia.org/resource/Constantinople
4,http://dbpedia.org/resource/Aeschylus,http://www.wikidata.org/prop/direct/P19,http://dbpedia.org/resource/Elefsina


## 4. Build a fixed prompt sample

For each true triple, the extractor processes the subject and object pages, plus outgoing links from those pages. This sampler uses that same one-hop idea, but caps links per entity page so the benchmark remains small.

A page is paired with the same normalized subject/object names used in the base extractor. Duplicate cleaned page content is skipped globally, as in its `processed_hashes` logic. The resulting prompt list is built **once** and reused unchanged for all models.


In [6]:
def relationship_prompt(entity1, entity2, page_content):
    # Kept aligned with LLMQuery.generate_relationship_prompt() from llm_query_final.py
    prompt = (
        f"Read the content below and write a natural paragraph using only the information that clearly involves both {entity1} and {entity2}. "
        f"Do not include anything unrelated to these two entities. "
        f"Write only in plain, fluent sentences without formatting, lists, subheadings, or commentary. "
        f"Only include clear and direct or indirect facts that connect or describe {entity1} and {entity2}.\\n\\n"
        f"Do not generate new text, omit information, or use any external knowledge."
        f"Content:\\n{page_content}\\n\\n"
        f"Do not miss or exclude any information that depicts a connection between {entity1} and {entity2}"
        f"Avoid phrases like 'there is no information about' or 'the only mention is'. "
        f"If the provided text does not mention a direct or indirect connection between {entity1} and {entity2} then output 'No Response'. "
    )
    return prompt.strip()


def build_prompt_sample(triples, n_prompts, n_triples_to_scan, max_links_per_entity_page, seed=42):
    rng = random.Random(seed)
    candidates = []
    seen_hashes = set()
    scanned = 0
    fetch_errors = []

    for subject_uri, _, object_uri in triples[:n_triples_to_scan]:
        scanned += 1
        subject_title = title_from_uri(subject_uri)
        object_title = title_from_uri(object_uri)
        entity1 = normalize_filename(subject_title)
        entity2 = normalize_filename(object_title)
        entity_pair = f"{entity1}_{entity2}"

        for entity_title in [subject_title, object_title]:
            try:
                page_text, links = fetch_wikipedia_content(
                    entity_title,
                    include_links=True,
                    max_links=max_links_per_entity_page
                )
            except Exception as exc:
                fetch_errors.append({"title": entity_title, "error": repr(exc)})
                continue

            if not page_text:
                continue

            # Main subject/object page candidate
            cleaned = clean_text(page_text)
            digest = hashlib.md5(cleaned.encode("utf-8")).hexdigest()
            if cleaned.strip() and digest not in seen_hashes:
                seen_hashes.add(digest)
                candidates.append({
                    "page_title": entity_title,
                    "page_type": "entity_page",
                    "entity1": entity1,
                    "entity2": entity2,
                    "entity_pair": entity_pair,
                    "cleaned_text": cleaned,
                    "content_md5": digest,
                })

            # One-hop linked pages, limited to a deterministic prefix
            for link_title in links[:max_links_per_entity_page]:
                try:
                    link_text, _ = fetch_wikipedia_content(
                        link_title,
                        include_links=False
                    )
                except Exception as exc:
                    fetch_errors.append({"title": link_title, "error": repr(exc)})
                    continue

                if not link_text:
                    continue
                cleaned_link = clean_text(link_text)
                link_digest = hashlib.md5(cleaned_link.encode("utf-8")).hexdigest()
                if not cleaned_link.strip() or link_digest in seen_hashes:
                    continue
                seen_hashes.add(link_digest)
                candidates.append({
                    "page_title": link_title,
                    "page_type": "linked_page",
                    "entity1": entity1,
                    "entity2": entity2,
                    "entity_pair": entity_pair,
                    "cleaned_text": cleaned_link,
                    "content_md5": link_digest,
                })

                if len(candidates) >= n_prompts * 3:
                    break

            if len(candidates) >= n_prompts * 3:
                break

        if len(candidates) >= n_prompts * 3:
            break

    # Randomize selection so the sample is not just the first few RDF entries.
    rng.shuffle(candidates)
    selected = candidates[:n_prompts]
    for i, item in enumerate(selected):
        item["prompt_id"] = f"P{i+1:03d}"
        item["prompt"] = relationship_prompt(
            item["entity1"], item["entity2"], item["cleaned_text"]
        )

    return selected, fetch_errors, scanned


prompts, fetch_errors, triples_scanned = build_prompt_sample(
    triples=triples,
    n_prompts=N_PROMPTS,
    n_triples_to_scan=N_TRIPLES_TO_SCAN,
    max_links_per_entity_page=MAX_LINKS_PER_ENTITY_PAGE,
    seed=RANDOM_SEED,
)

print(f"Scanned {triples_scanned} triples; collected {len(prompts)} prompts.")
print(f"Wikipedia fetch errors: {len(fetch_errors)}")
if len(prompts) < N_PROMPTS:
    print("WARNING: fewer prompts than requested. Increase N_TRIPLES_TO_SCAN or MAX_LINKS_PER_ENTITY_PAGE.")

sample_df = pd.DataFrame([
    {
        "prompt_id": p["prompt_id"],
        "page_title": p["page_title"],
        "page_type": p["page_type"],
        "entity_pair": p["entity_pair"],
        "input_chars": len(p["cleaned_text"]),
        "input_words": len(p["cleaned_text"].split()),
    }
    for p in prompts
])
display(sample_df)


Scanned 6 triples; collected 30 prompts.
Wikipedia fetch errors: 0


,prompt_id,page_title,page_type,entity_pair,input_chars,input_words
0,P001,Adhan,linked_page,adam mickiewicz_constantinople,20422,3327
1,P002,Winnipeg,entity_page,a. e. van vogt_winnipeg,41032,6406
2,P003,Lithuania_Governorate,entity_page,adam mickiewicz_lithuania governorate,846,126
3,P004,20th Century Fox,linked_page,a. e. van vogt_winnipeg,56952,9008
4,P005,Aeolic Greek,linked_page,aeschylus_elefsina,9874,1444
5,P006,Acarnanian League,linked_page,aeschylus_elefsina,4302,672
6,P007,1951 Pan American Games,linked_page,a. e. van vogt_winnipeg,2111,342
7,P008,Canal Hollywood,linked_page,a. e. van vogt_hollywood,837,137
8,P009,"Adam Mickiewicz Monument, Kraków",linked_page,adam mickiewicz_lithuania governorate,2504,405
9,P010,Adam_Mickiewicz,entity_page,adam mickiewicz_lithuania governorate,34731,5461


In [7]:
fetch_errors

[]

## 5. Run the benchmark

Every model receives exactly the same prompt set, in the same order. Requests are run sequentially to keep the benchmark simple and avoid competing for GPU memory.

The notebook records:
- wall-clock latency for each call (including any model load time);
- Ollama's prompt and generated token counts;
- prompt evaluation and generation durations reported by Ollama;
- output length and completion reason;
- errors and empty responses.

`NUM_PREDICT` is placed under `options`, which is where Ollama expects generation options. If `done_reason == "length"`, the response may have been cut off; treat that separately when interpreting quality and timings.


In [8]:
def run_one_call(model, prompt, prompt_id, think_mode):
    payload = {
        "model": model,
        "prompt": prompt,
        "stream": False,
        "keep_alive": KEEP_ALIVE,
        "options": {
            "temperature": TEMPERATURE,
            "num_predict": NUM_PREDICT,
        },
    }
    if NUM_CTX is not None:
        payload["options"]["num_ctx"] = NUM_CTX
    if think_mode is not None:
        payload["think"] = think_mode

    started = time.perf_counter()
    timestamp = datetime.now(timezone.utc).isoformat()
    try:
        response = requests.post(
            API_URL,
            json=payload,
            headers=API_HEADERS,
            timeout=(30, 60 * 60),
        )
        response.raise_for_status()
        result = response.json()
        elapsed = time.perf_counter() - started

        final_text = result.get("response") or ""
        thinking_text = result.get("thinking") or ""

        return {
            "timestamp_utc": timestamp,
            "model": model,
            "prompt_id": prompt_id,
            "status": "ok",
            "wall_seconds": elapsed,
            "prompt_tokens": result.get("prompt_eval_count"),
            "generated_tokens": result.get("eval_count"),
            "prompt_eval_seconds": (
                result.get("prompt_eval_duration", 0) / 1e9
                if result.get("prompt_eval_duration") is not None else None
            ),
            "generation_seconds": (
                result.get("eval_duration", 0) / 1e9
                if result.get("eval_duration") is not None else None
            ),
            "load_seconds": (
                result.get("load_duration", 0) / 1e9
                if result.get("load_duration") is not None else None
            ),
            "done": result.get("done"),
            "done_reason": result.get("done_reason"),
            "output_chars": len(final_text),
            "thinking_chars": len(thinking_text),
            "output_text": final_text,
            "error": None,
        }
    except Exception as exc:
        elapsed = time.perf_counter() - started
        return {
            "timestamp_utc": timestamp,
            "model": model,
            "prompt_id": prompt_id,
            "status": "error",
            "wall_seconds": elapsed,
            "prompt_tokens": None,
            "generated_tokens": None,
            "prompt_eval_seconds": None,
            "generation_seconds": None,
            "load_seconds": None,
            "done": None,
            "done_reason": None,
            "output_chars": 0,
            "thinking_chars": 0,
            "output_text": "",
            "error": repr(exc),
        }


# Safety check before a potentially long run.
if not prompts:
    raise RuntimeError("No prompts were collected. Check INPUT_FILE and the sampling configuration.")

print(f"About to run {len(MODELS)} models × {len(prompts)} prompts = {len(MODELS) * len(prompts)} calls.")
print("Models:", MODELS)


About to run 6 models × 30 prompts = 180 calls.
Models: ['qwen3.5:2b', 'qwen3.5:4b', 'gemma4:e2b', 'gemma4:e4b', 'qwen3.5:9b', 'deepseek-r1:14b']


In [9]:
# Run all model/prompt combinations.
# Results are checkpointed after every call, so an interrupted run still leaves partial data.
records = []
results_csv = RESULTS_DIR / f"call_results_{RUN_ID}.csv"
results_jsonl = RESULTS_DIR / f"call_results_{RUN_ID}.jsonl"


for model_index, model in enumerate(MODELS, start=1):
    for think_mode in THINK_MODES:
        mode_name = "reasoning_on" if think_mode else "reasoning_off"

        print(
            f"\n[{model_index}/{len(MODELS)}] "
            f"Benchmarking {model} | {mode_name}"
        )

        for prompt_index, item in enumerate(prompts, start=1):
            print(
                f"  {prompt_index}/{len(prompts)}: "
                f"{item['prompt_id']} – {item['page_title']}",
                end=" ... "
            )

            record = run_one_call(
                model,
                item["prompt"],
                item["prompt_id"],
                think_mode
            )

            record["think_mode"] = mode_name
            record["page_title"] = item["page_title"]
            record["page_type"] = item["page_type"]
            record["entity_pair"] = item["entity_pair"]
            record["input_chars"] = len(item["cleaned_text"])
            record["input_words"] = len(item["cleaned_text"].split())

            records.append(record)

            pd.DataFrame(records).to_csv(results_csv, index=False)

            with open(results_jsonl, "a", encoding="utf-8") as f:
                f.write(json.dumps(record, ensure_ascii=False) + "\n")

            if record["status"] == "ok":
                print(
                    f"{record['wall_seconds']:.1f}s, "
                    f"{record['generated_tokens']} tokens, "
                    f"{record['done_reason']}"
                )
            else:
                print("ERROR:", record["error"])

results = pd.DataFrame(records)
display(results.head())

display(results.head())



[1/6] Benchmarking qwen3.5:2b | reasoning_on
  1/30: P001 – Adhan ... 15.8s, 4096 tokens, length
  2/30: P002 – Winnipeg ... 11.2s, 2950 tokens, stop
  3/30: P003 – Lithuania_Governorate ... 7.3s, 2191 tokens, stop
  4/30: P004 – 20th Century Fox ... 15.4s, 4096 tokens, length
  5/30: P005 – Aeolic Greek ... 14.0s, 4096 tokens, length
  6/30: P006 – Acarnanian League ... 9.3s, 2772 tokens, stop
  7/30: P007 – 1951 Pan American Games ... 6.8s, 1971 tokens, stop
  8/30: P008 – Canal Hollywood ... 9.6s, 2873 tokens, stop
  9/30: P009 – Adam Mickiewicz Monument, Kraków ... 11.9s, 3504 tokens, stop
  10/30: P010 – Adam_Mickiewicz ... 14.6s, 4096 tokens, length
  11/30: P011 – Amur Oblast (Russian Empire) ... 13.8s, 4096 tokens, length
  12/30: P012 – Arkhangelsk Governorate ... 8.7s, 2530 tokens, stop
  13/30: P013 – Achaean League ... 14.3s, 4096 tokens, length
  14/30: P014 – 1967 Pan American Games ... 8.0s, 2296 tokens, stop
  15/30: P015 – Adile Sultan Palace ... 10.2s, 3038 tokens, s

,timestamp_utc,model,prompt_id,status,wall_seconds,prompt_tokens,generated_tokens,prompt_eval_seconds,generation_seconds,load_seconds,...,output_chars,thinking_chars,output_text,error,think_mode,page_title,page_type,entity_pair,input_chars,input_words
0,2026-11-15T11:17:37.181006+00:00,qwen3.5:2b,P001,ok,15.807025,5534,4096,0.300045,13.708665,1.566605,...,0,16730,,None,reasoning_on,Adhan,linked_page,adam mickiewicz_constantinople,20422,3327
1,2026-11-15T11:17:53.010786+00:00,qwen3.5:2b,P002,ok,11.221468,9790,2950,0.434389,10.415540,0.001049,...,11,11479,No Response,None,reasoning_on,Winnipeg,entity_page,a. e. van vogt_winnipeg,41032,6406
2,2026-11-15T11:18:04.239016+00:00,qwen3.5:2b,P003,ok,7.335445,373,2191,0.023887,7.106558,0.000788,...,11,9413,No Response,None,reasoning_on,Lithuania_Governorate,entity_page,adam mickiewicz_lithuania governorate,846,126
3,2026-11-15T11:18:11.586663+00:00,qwen3.5:2b,P004,ok,15.428724,13699,4096,0.637119,14.370529,0.000984,...,0,15579,,None,reasoning_on,20th Century Fox,linked_page,a. e. van vogt_winnipeg,56952,9008
4,2026-11-15T11:18:27.020703+00:00,qwen3.5:2b,P005,ok,14.027765,3596,4096,0.172179,13.528593,0.000754,...,0,14221,,None,reasoning_on,Aeolic Greek,linked_page,aeschylus_elefsina,9874,1444


,timestamp_utc,model,prompt_id,status,wall_seconds,prompt_tokens,generated_tokens,prompt_eval_seconds,generation_seconds,load_seconds,...,output_chars,thinking_chars,output_text,error,think_mode,page_title,page_type,entity_pair,input_chars,input_words
0,2026-11-15T11:17:37.181006+00:00,qwen3.5:2b,P001,ok,15.807025,5534,4096,0.300045,13.708665,1.566605,...,0,16730,,None,reasoning_on,Adhan,linked_page,adam mickiewicz_constantinople,20422,3327
1,2026-11-15T11:17:53.010786+00:00,qwen3.5:2b,P002,ok,11.221468,9790,2950,0.434389,10.415540,0.001049,...,11,11479,No Response,None,reasoning_on,Winnipeg,entity_page,a. e. van vogt_winnipeg,41032,6406
2,2026-11-15T11:18:04.239016+00:00,qwen3.5:2b,P003,ok,7.335445,373,2191,0.023887,7.106558,0.000788,...,11,9413,No Response,None,reasoning_on,Lithuania_Governorate,entity_page,adam mickiewicz_lithuania governorate,846,126
3,2026-11-15T11:18:11.586663+00:00,qwen3.5:2b,P004,ok,15.428724,13699,4096,0.637119,14.370529,0.000984,...,0,15579,,None,reasoning_on,20th Century Fox,linked_page,a. e. van vogt_winnipeg,56952,9008
4,2026-11-15T11:18:27.020703+00:00,qwen3.5:2b,P005,ok,14.027765,3596,4096,0.172179,13.528593,0.000754,...,0,14221,,None,reasoning_on,Aeolic Greek,linked_page,aeschylus_elefsina,9874,1444


## 6. Summarize performance and estimate full-run duration

The estimate uses the measured **wall-clock time per successful call**. It reports both:
- mean latency, which is sensitive to long calls and is useful for total-time planning;
- median latency, which is more robust to outliers.

The first call for a model can include model loading. The summary therefore also reports first-call and later-call averages. For a full run, use the later-call estimate if the model will remain loaded; use the all-call estimate if the server frequently unloads models.

Enter the expected number of LLM calls **per model**. If the current working estimate is 400,000 calls per model, leave it as-is. This is a simple linear extrapolation; it does not assume extra speedup from batching or parallelism.


In [10]:
EXPECTED_CALLS_PER_MODEL = 400_000  # replace with a better count after the sample run

def safe_mean(series):
    values = pd.to_numeric(series, errors="coerce").dropna()
    return float(values.mean()) if len(values) else None

summary_rows = []
for model in MODELS:
    for think_mode in THINK_MODES:
        mode_name = "reasoning_on" if think_mode else "reasoning_off"
        group = results[
            (results["model"] == model) &
            (results["think_mode"] == mode_name)
        ]
        ok = group[group["status"] == "ok"].copy()
        later = ok.iloc[1:] if len(ok) > 1 else ok.iloc[0:0]

        mean_all = safe_mean(ok["wall_seconds"])
        mean_later = safe_mean(later["wall_seconds"])
        median_all = float(ok["wall_seconds"].median()) if len(ok) else None
        success_rate = len(ok) / len(group) if len(group) else 0

        chosen_mean = mean_later if mean_later is not None else mean_all
        estimated_hours = (
            chosen_mean * EXPECTED_CALLS_PER_MODEL / 3600
            if chosen_mean is not None else None
        )

        summary_rows.append({
            "model": model,
            "calls_attempted": len(group),
            "successful_calls": len(ok),
            "success_rate": success_rate,
            "mean_wall_s_all": mean_all,
            "mean_wall_s_after_first": mean_later,
            "median_wall_s": median_all,
            "total_wall_min": float(group["wall_seconds"].sum()) / 60,
            "mean_prompt_tokens": safe_mean(ok["prompt_tokens"]),
            "mean_generated_tokens": safe_mean(ok["generated_tokens"]),
            "mean_generation_s": safe_mean(ok["generation_seconds"]),
            "mean_load_s": safe_mean(ok["load_seconds"]),
            "estimated_hours_per_model": estimated_hours,
            "estimated_days_per_model": estimated_hours / 24 if estimated_hours is not None else None,
        })

summary = pd.DataFrame(summary_rows)
display(summary.round(3))

valid_hours = summary["estimated_hours_per_model"].dropna()
if len(valid_hours):
    print(f"Estimated sequential runtime for all {len(valid_hours)} models: {valid_hours.sum():,.1f} GPU-hours")
    print(f"That is {valid_hours.sum()/24:,.1f} days if run continuously on one GPU.")


,model,calls_attempted,successful_calls,success_rate,mean_wall_s_all,mean_wall_s_after_first,median_wall_s,total_wall_min,mean_prompt_tokens,mean_generated_tokens,mean_generation_s,mean_load_s,estimated_hours_per_model,estimated_days_per_model
0,qwen3.5:2b,30,30,1.0,11.861,11.725,13.088,5.930,4252.133,3372.467,11.326,0.053,1302.754,54.281
1,qwen3.5:2b,30,30,1.0,0.403,0.404,0.366,0.202,4254.133,28.967,0.097,0.001,44.918,1.872
2,qwen3.5:4b,30,30,1.0,7.633,7.669,5.729,3.817,4252.133,1464.133,6.755,0.087,852.134,35.506
3,qwen3.5:4b,30,30,1.0,0.910,0.912,0.529,0.455,4254.133,30.600,0.142,0.001,101.305,4.221
4,gemma4:e2b,30,30,1.0,1.488,1.351,1.069,0.744,4136.433,357.833,0.761,0.131,150.118,6.255
5,gemma4:e2b,30,30,1.0,0.665,0.661,0.437,0.332,4129.433,31.000,0.066,0.001,73.442,3.060
6,gemma4:e4b,30,30,1.0,2.066,1.946,1.559,1.033,4136.433,364.433,1.021,0.115,216.213,9.009
7,gemma4:e4b,30,30,1.0,0.958,0.954,0.658,0.479,4129.433,8.267,0.025,0.001,106.013,4.417
8,qwen3.5:9b,30,30,1.0,13.039,12.926,9.283,6.519,4252.133,1836.333,11.959,0.096,1436.262,59.844
9,qwen3.5:9b,30,30,1.0,1.082,1.080,0.632,0.541,4254.133,18.533,0.117,0.001,120.036,5.002


Estimated sequential runtime for all 12 models: 5,767.1 GPU-hours
That is 240.3 days if run continuously on one GPU.


## 7. Inspect truncation, failures, and output examples

A large `thinking_chars` value with an empty `output_text`, especially when `done_reason` is `length`, indicates the model may have spent its generation budget reasoning without producing a final answer. Do not count that as a successful usable simplification.

Also inspect `done_reason == "length"` for non-empty responses: those may be truncated.


In [11]:
if len(results):
    # Summary: model × reasoning mode × status × completion reason
    display(
        results.groupby(
            ["model", "think_mode", "status", "done_reason"],
            dropna=False
        )
        .size()
        .rename("count")
        .reset_index()
    )

    # Calls that reached the generation token limit
    truncated = results[results["done_reason"] == "length"][
        ["model", "think_mode", "prompt_id", "page_title",
         "wall_seconds", "prompt_tokens", "generated_tokens",
         "thinking_chars", "output_chars"]
    ]

    print("Calls ending because of token limit:", len(truncated))
    display(truncated)

    # Failed calls
    errors = results[results["status"] == "error"][
        ["model", "think_mode", "prompt_id", "page_title", "error"]
    ]

    print("Failed calls:", len(errors))
    display(errors)

    # Show one output per model and reasoning mode
    for model in MODELS:
        for think_mode in THINK_MODES:
            example = results[
                (results["model"] == model) &
                (results["think_mode"] == ("reasoning_on" if think_mode else "reasoning_off")) &
                (results["output_text"].fillna("") != "")
            ].head(1)

            if len(example):
                row = example.iloc[0]
                mode_label = "reasoning ON" if think_mode else "reasoning OFF"

                print(
                    f"\n--- {model} / {mode_label} / "
                    f"{row['prompt_id']} / {row['page_title']} ---"
                )
                print(row["output_text"][:2500])

,model,think_mode,status,done_reason,count
0,deepseek-r1:14b,reasoning_off,ok,stop,30
1,deepseek-r1:14b,reasoning_on,ok,stop,30
2,gemma4:e2b,reasoning_off,ok,stop,30
3,gemma4:e2b,reasoning_on,ok,stop,30
4,gemma4:e4b,reasoning_off,ok,stop,30
5,gemma4:e4b,reasoning_on,ok,stop,30
6,qwen3.5:2b,reasoning_off,ok,stop,30
7,qwen3.5:2b,reasoning_on,ok,length,15
8,qwen3.5:2b,reasoning_on,ok,stop,15
9,qwen3.5:4b,reasoning_off,ok,stop,30


Calls ending because of token limit: 23


,model,think_mode,prompt_id,page_title,wall_seconds,prompt_tokens,generated_tokens,thinking_chars,output_chars
0,qwen3.5:2b,reasoning_on,P001,Adhan,15.807025,5534,4096,16730,0
3,qwen3.5:2b,reasoning_on,P004,20th Century Fox,15.428724,13699,4096,15579,0
4,qwen3.5:2b,reasoning_on,P005,Aeolic Greek,14.027765,3596,4096,14221,0
9,qwen3.5:2b,reasoning_on,P010,Adam_Mickiewicz,14.642896,8402,4096,16595,0
10,qwen3.5:2b,reasoning_on,P011,Amur Oblast (Russian Empire),13.778662,2368,4096,17546,0
12,qwen3.5:2b,reasoning_on,P013,Achaean League,14.311240,6414,4096,13231,0
17,qwen3.5:2b,reasoning_on,P018,Aegisthus,13.705739,1800,4096,15771,0
19,qwen3.5:2b,reasoning_on,P020,Acritic songs,13.858976,1737,4096,17198,0
21,qwen3.5:2b,reasoning_on,P022,Gela,14.575538,3511,4096,15516,0
22,qwen3.5:2b,reasoning_on,P023,Aegina,14.859267,9167,4096,14343,0


Failed calls: 0


,model,think_mode,prompt_id,page_title,error



--- qwen3.5:2b / reasoning ON / P002 / Winnipeg ---
No Response

--- qwen3.5:2b / reasoning OFF / P001 / Adhan ---
There is no response because the provided text contains no information regarding Adam Mickiewicz or Constantinople.

--- qwen3.5:4b / reasoning ON / P001 / Adhan ---
No Response

--- qwen3.5:4b / reasoning OFF / P001 / Adhan ---
No Response.

--- gemma4:e2b / reasoning ON / P001 / Adhan ---
No Response

--- gemma4:e2b / reasoning OFF / P001 / Adhan ---
The provided text does not contain any information that clearly involves both Adam Mickiewicz and Constantinople.

--- gemma4:e4b / reasoning ON / P001 / Adhan ---
No Response

--- gemma4:e4b / reasoning OFF / P001 / Adhan ---
No Response

--- qwen3.5:9b / reasoning ON / P001 / Adhan ---
No Response

--- qwen3.5:9b / reasoning OFF / P001 / Adhan ---
No Response

--- deepseek-r1:14b / reasoning ON / P001 / Adhan ---
No Response

--- deepseek-r1:14b / reasoning OFF / P001 / Adhan ---
<think>
Okay, so I need to figure out how 

## 8. Optional: benchmark Maverick coreference separately

In the base extractor, Maverick runs **after each LLM response**, including responses such as `No Response`. Its runtime can therefore be a substantial additional cost. This optional section times coreference on a few available model outputs.

Run this only if the environment has `maverick` installed and the model weights are available. The device defaults to CUDA when available; set `COREF_DEVICE=cpu` or another supported device before launching Jupyter if needed.

This measures coreference on the generated outputs, not on the original Wikipedia text. The original extractor passes the LLM output to `compute_coref()`.


In [12]:
RUN_COREF_BENCHMARK = False
COREF_EXAMPLES_PER_MODEL = 3

if RUN_COREF_BENCHMARK:
    import torch
    from maverick import Maverick

    coref_device = os.environ.get(
        "COREF_DEVICE",
        "cuda:0" if torch.cuda.is_available() else "cpu"
    )
    print("Maverick device:", coref_device)
    coref_model = Maverick(
        hf_name_or_path="sapienzanlp/maverick-mes-ontonotes",
        device=coref_device
    )

    coref_records = []
    usable = results[
        (results["status"] == "ok") &
        (results["output_text"].fillna("") != "")
    ]

    for model in MODELS:
        subset = usable[usable["model"] == model].head(COREF_EXAMPLES_PER_MODEL)
        for _, row in subset.iterrows():
            start = time.perf_counter()
            try:
                _ = coref_model.predict(row["output_text"])
                elapsed = time.perf_counter() - start
                coref_records.append({
                    "model": model,
                    "prompt_id": row["prompt_id"],
                    "coref_seconds": elapsed,
                    "status": "ok",
                    "error": None,
                })
            except Exception as exc:
                elapsed = time.perf_counter() - start
                coref_records.append({
                    "model": model,
                    "prompt_id": row["prompt_id"],
                    "coref_seconds": elapsed,
                    "status": "error",
                    "error": repr(exc),
                })

    coref_results = pd.DataFrame(coref_records)
    coref_path = RESULTS_DIR / f"coref_results_{RUN_ID}.csv"
    coref_results.to_csv(coref_path, index=False)
    display(coref_results)
    display(coref_results.groupby("model")["coref_seconds"].agg(["mean", "median", "count"]))
else:
    print("Coreference benchmark skipped. Set RUN_COREF_BENCHMARK = True to enable it.")


Coreference benchmark skipped. Set RUN_COREF_BENCHMARK = True to enable it.


## 9. Export a compact report

This saves the model summary and the exact prompt sample metadata alongside the call-level CSV/JSONL files. Keep these files with the experiment notes so the runtime estimate can be reproduced.


In [13]:
summary_path = RESULTS_DIR / f"model_summary_{RUN_ID}.csv"
sample_path = RESULTS_DIR / f"prompt_sample_{RUN_ID}.csv"
summary.to_csv(summary_path, index=False)
sample_df.to_csv(sample_path, index=False)

ts = pd.to_datetime(results["timestamp_utc"])
run_started = ts.min()
run_finished = (ts + pd.to_timedelta(results["wall_seconds"], unit="s")).max()
run_total_min = (run_finished - run_started).total_seconds() / 60
time_by_model = (
    results.groupby(["model", "think_mode"])["wall_seconds"].sum().div(60).round(1)
)
print(f"Całkowity czas przebiegu: {run_total_min:.1f} min ({run_total_min/60:.2f} h)")
display(time_by_model.rename("minuty").reset_index())


metadata = {
    "run_id": RUN_ID,
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "input_file": str(INPUT_FILE),
    "models": MODELS,
    "n_prompts": len(prompts),
    "triples_scanned": triples_scanned,
    "max_links_per_entity_page": MAX_LINKS_PER_ENTITY_PAGE,
    "temperature": TEMPERATURE,
    "num_predict": NUM_PREDICT,
    "num_ctx": NUM_CTX,
    "think": THINK_MODES,
    "expected_calls_per_model_for_estimate": EXPECTED_CALLS_PER_MODEL,
    "api_endpoint": API_URL,
     "run_started_utc": run_started.isoformat(),
    "run_finished_utc": run_finished.isoformat(),
    "run_total_minutes": round(run_total_min, 1),
    "minutes_by_model_mode": {f"{m} | {t}": v for (m, t), v in time_by_model.items()},
    "notes": "Runtime estimates are linear extrapolations from a small sample; no batching speedup assumed."
}
metadata_path = RESULTS_DIR / f"run_metadata_{RUN_ID}.json"
metadata_path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")

print("Saved report files:")
for path in [results_csv, results_jsonl, summary_path, sample_path, metadata_path]:
    print(" -", path)


Całkowity czas przebiegu: 26.3 min (0.44 h)


,model,think_mode,minuty
0,deepseek-r1:14b,reasoning_off,3.0
1,deepseek-r1:14b,reasoning_on,3.2
2,gemma4:e2b,reasoning_off,0.3
3,gemma4:e2b,reasoning_on,0.7
4,gemma4:e4b,reasoning_off,0.5
5,gemma4:e4b,reasoning_on,1.0
6,qwen3.5:2b,reasoning_off,0.2
7,qwen3.5:2b,reasoning_on,5.9
8,qwen3.5:4b,reasoning_off,0.5
9,qwen3.5:4b,reasoning_on,3.8


Saved report files:
 - /Users/michal/university/sem7/semantic-web/no-need-to-be-so-big/results/model_benchmark/call_results_20261115_121340.csv
 - /Users/michal/university/sem7/semantic-web/no-need-to-be-so-big/results/model_benchmark/call_results_20261115_121340.jsonl
 - /Users/michal/university/sem7/semantic-web/no-need-to-be-so-big/results/model_benchmark/model_summary_20261115_121340.csv
 - /Users/michal/university/sem7/semantic-web/no-need-to-be-so-big/results/model_benchmark/prompt_sample_20261115_121340.csv
 - /Users/michal/university/sem7/semantic-web/no-need-to-be-so-big/results/model_benchmark/run_metadata_20261115_121340.json


## Interpretation checklist

Before using the extrapolated runtime in the project report, check:

- **Sample size:** 20–50 prompts is a first estimate; 100+ varied prompts is better if feasible.
- **Representativeness:** the sample should include both entity pages and linked pages, and short and long articles.
- **Cold start:** first-call latency may include loading weights. The later-call average is usually more representative when a model stays loaded.
- **Token limits:** investigate `done_reason == "length"` and empty final outputs before accepting the timing/quality results.
- **Hardware:** record GPU model, VRAM, RAM, Ollama version, model quantization/tag, and whether the model was fully or partly offloaded to CPU.
- **Concurrency:** this notebook runs one request at a time. Batching or concurrent inference may improve throughput, but benchmark that separately rather than assuming a speedup.
- **Coreference:** include the separate Maverick estimate if estimating the full pipeline.
- **Actual call count:** 400,000 calls per model is an assumption, not something this notebook verifies. The extractor skips missing pages and globally duplicated cleaned content, so the real count may be lower. Conversely, the count depends on the number of true triples and linked pages processed.

For a stronger estimate, run this notebook once with a small sample, then repeat with a larger sample on the final experiment machine.
